In [1]:
import torch
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import os
from dotenv import load_dotenv
import cv2
import glob
from random import randint
import lightning as L
import sys

from torch.nn.utils.rnn import pad_sequence

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor

In [2]:
load_dotenv()
GESTURE_VIDEOS_PATH = os.getenv("GESTURE_VIDEOS")

In [3]:
# Dataset
df = pd.read_csv(f"{GESTURE_VIDEOS_PATH}/metadata.csv")
df.head()

,Video Name,Frames,Sex,Hand,Background,Illumination,People in Scene,Background Motion,Set,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,1CM1_4_R__229,3751,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1CM1_4_R__230,3684,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1CM1_4_R__231,3747,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1CM1_4_R__232,3858,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1CM42_11_R__205,3686,M,Right,Plain,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
file = open(f"{GESTURE_VIDEOS_PATH}/annotations/annotations/class_details.txt")
text = file.read().split('\n')
label_id_dict = dict()
for row in text:
    info = row.split('\t')[:3]
    label_id_dict[info[0]] = info[1:] 

In [5]:
label_id_dict

{'id': ['Label', 'Gesture'],
 '1': ['D0X', 'Non-gesture'],
 '2': ['B0A', 'Pointing with one finger'],
 '3': ['B0B', 'Pointing with two fingers'],
 '4': ['G01', 'Click with one finger'],
 '5': ['G02', 'Click with two fingers'],
 '6': ['G03', 'Throw up'],
 '7': ['G04', 'Throw down'],
 '8': ['G05', 'Throw left'],
 '9': ['G06', 'Throw right'],
 '10': ['G07', 'Open twice'],
 '11': ['G08', 'Double click with one finger'],
 '12': ['G09', 'Double click with two fingers'],
 '13': ['G10', 'Zoom in'],
 '14': ['G11', 'Zoom out'],
 '': []}

In [6]:
file = open(f"{GESTURE_VIDEOS_PATH}/annotations/annotations/Annot_List.txt")
text = file.read().split('\n')
video_info_dict = dict()
for row in text:
    info = row.split(',')
    try:
        video_info_dict[info[0]].append(info[1:]) 
    except:
        video_info_dict[info[0]] = []
        video_info_dict[info[0]].append(info[1:]) 


In [7]:
video_info_dict

{'video': [['label', 'id', 't_start', 't_end', 'frames']],
 '1CM1_4_R__229': [['D0X', '1', '1', '17', '17'],
  ['G11', '14', '18', '55', '38'],
  ['B0B', '3', '56', '284', '229'],
  ['G04', '7', '285', '308', '24'],
  ['B0B', '3', '309', '502', '194'],
  ['G05', '8', '503', '544', '42'],
  ['B0B', '3', '545', '857', '313'],
  ['G03', '6', '858', '899', '42'],
  ['B0A', '2', '900', '1122', '223'],
  ['D0X', '1', '1123', '1432', '310'],
  ['G02', '5', '1433', '1457', '25'],
  ['B0A', '2', '1458', '1707', '250'],
  ['D0X', '1', '1708', '2042', '335'],
  ['G08', '11', '2043', '2077', '35'],
  ['B0A', '2', '2078', '2350', '273'],
  ['G06', '9', '2351', '2391', '41'],
  ['B0B', '3', '2392', '2603', '212'],
  ['G10', '13', '2604', '2646', '43'],
  ['B0B', '3', '2647', '2794', '148'],
  ['D0X', '1', '2795', '2993', '199'],
  ['G09', '12', '2994', '3029', '36'],
  ['B0A', '2', '3030', '3232', '203'],
  ['G07', '10', '3233', '3277', '45'],
  ['B0A', '2', '3278', '3645', '368'],
  ['G01', '4', '3

In [8]:
# Correct info dicts
new_label_id_dict = {}
new_video_info_dict = {}

for k, v in label_id_dict.items():
    if k.isdigit() and len(v) > 0:
        new_label_id_dict[f"{int(k) - 1}"] = v

label_id_dict = new_label_id_dict

for k, v in video_info_dict.items():
    for idx, video_fragment in enumerate(v):
        label, id, t_start, t_end, frames = video_fragment
        if id.isdigit():
            new_video_info_dict[(k, idx)] = [label, f"{int(id) - 1}", t_start, t_end, frames]

video_info_dict = new_video_info_dict

In [9]:
label_id_dict

{'0': ['D0X', 'Non-gesture'],
 '1': ['B0A', 'Pointing with one finger'],
 '2': ['B0B', 'Pointing with two fingers'],
 '3': ['G01', 'Click with one finger'],
 '4': ['G02', 'Click with two fingers'],
 '5': ['G03', 'Throw up'],
 '6': ['G04', 'Throw down'],
 '7': ['G05', 'Throw left'],
 '8': ['G06', 'Throw right'],
 '9': ['G07', 'Open twice'],
 '10': ['G08', 'Double click with one finger'],
 '11': ['G09', 'Double click with two fingers'],
 '12': ['G10', 'Zoom in'],
 '13': ['G11', 'Zoom out']}

In [10]:
video_info_dict

{('1CM1_4_R__229', 0): ['D0X', '0', '1', '17', '17'],
 ('1CM1_4_R__229', 1): ['G11', '13', '18', '55', '38'],
 ('1CM1_4_R__229', 2): ['B0B', '2', '56', '284', '229'],
 ('1CM1_4_R__229', 3): ['G04', '6', '285', '308', '24'],
 ('1CM1_4_R__229', 4): ['B0B', '2', '309', '502', '194'],
 ('1CM1_4_R__229', 5): ['G05', '7', '503', '544', '42'],
 ('1CM1_4_R__229', 6): ['B0B', '2', '545', '857', '313'],
 ('1CM1_4_R__229', 7): ['G03', '5', '858', '899', '42'],
 ('1CM1_4_R__229', 8): ['B0A', '1', '900', '1122', '223'],
 ('1CM1_4_R__229', 9): ['D0X', '0', '1123', '1432', '310'],
 ('1CM1_4_R__229', 10): ['G02', '4', '1433', '1457', '25'],
 ('1CM1_4_R__229', 11): ['B0A', '1', '1458', '1707', '250'],
 ('1CM1_4_R__229', 12): ['D0X', '0', '1708', '2042', '335'],
 ('1CM1_4_R__229', 13): ['G08', '10', '2043', '2077', '35'],
 ('1CM1_4_R__229', 14): ['B0A', '1', '2078', '2350', '273'],
 ('1CM1_4_R__229', 15): ['G06', '8', '2351', '2391', '41'],
 ('1CM1_4_R__229', 16): ['B0B', '2', '2392', '2603', '212'],
 (

In [11]:
num_videos = len(glob.glob(f'{GESTURE_VIDEOS_PATH}/videos/videos/*'))
print(len(video_info_dict))
print(num_videos)

5649
200


In [12]:
class IPNData(Dataset):
    def __init__(self, video_info):
        self.paths = glob.glob(f'{GESTURE_VIDEOS_PATH}/videos/videos/*')
        self.video_info = video_info
        
    def get_frames(self, path, info):
        frames = []
        label, ID, start_frame, end_frame, num_frames = info
        
        start_frame = int(start_frame)
        end_frame = int(end_frame)
        frame_no = start_frame

        cap = cv2.VideoCapture(str(path))
        cap.set(cv2.CAP_PROP_POS_FRAMES, 5)
        ret, frame = cap.read()

        # Loop over the frames
        while ret:

            if start_frame <= frame_no <= end_frame:
                frames.append(frame)

            frame_no += 1


            # Read the next frame
            ret, frame = cap.read()

        # Release the video capture
        cap.release()
        return frames
    
    def __getitem__(self, video_ix, section_ix):
        path = self.paths[video_ix]
        video_name = str(path).split('/')[-1][:-4]
        info = self.video_info[(video_name, section_ix)]
        label, ID, start_frame, end_frame, num_frames = info
        frames = self.get_frames(path, info)
        return frames, label, ID
    
    def __len__(self):
        return len(self.paths)
    
    def choose(self): return self[randint(len(self))]

In [13]:
ds_raw = IPNData(video_info=video_info_dict)
mpp = MediaPipeProcessor()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772920987.099504   10869 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772920987.109195   10877 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
label_to_index = {}

for k in label_id_dict:
    label, gesture = label_id_dict[k]
    label_to_index[label] = int(k)

In [ ]:
N_V = num_videos
T = 1
ds = []
train_size = int(0.9 * (N_V - T))
val_size = (N_V -T) - train_size

for i in range(N_V):
    for j in range(sys.maxsize):
        try:
            frames, label, ID = ds_raw.__getitem__(i, j)

            keypoint_sequence = mpp.process_video(frames)

            ds.append([keypoint_sequence, label_to_index[label]]) # using ID as label
        except:
            break

test_ds = ds[-1]
train_val_ds = ds[:-1]

train_ds, val_ds = random_split(train_val_ds, [train_size, val_size])

In [20]:
print(len(train_ds))
print(len(val_ds))

179
20


In [21]:
print(test_ds)

[array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(60, 126)), 0]


In [22]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [24]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = len(label_id_dict)

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [25]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 372, 126]) y: torch.Size([32]) logits: torch.Size([32, 14])


In [26]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model   | KeypointTransformer | 1.3 M  | train | 0    
1 | loss_fn | CrossEntropyLoss    | 0      | train | 0    
----------------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params


Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (6) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 19: 100%|██████████| 6/6 [00:30<00:00,  0.20it/s, v_num=13, train_loss_step=0.903, val_loss=0.461, train_loss_epoch=0.546]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 6/6 [00:30<00:00,  0.20it/s, v_num=13, train_loss_step=0.903, val_loss=0.461, train_loss_epoch=0.546]


In [27]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [28]:
#Testing model

In [29]:
test_ds[0]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(60, 126))

In [30]:
keypoint_sequence, label = test_ds

x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

# prediction
with torch.no_grad():
    logits = model(x)
    predicted_idx = torch.argmax(logits, dim=-1).item()

print(f"Prediction: {label_id_dict[f"{predicted_idx}"]}")
print(f"Reality: {label_id_dict[f"{label}"]}")

Prediction: ['D0X', 'Non-gesture']
Reality: ['D0X', 'Non-gesture']


In [31]:
for i in range(150):    
    keypoint_sequence, label = train_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    print(f"Prediction: {predicted_idx}")
    print(f"Reality: {label}")

Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 2
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 3
Reality: 8
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 13
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0
Prediction: 0
Reality: 5
Prediction: 0
Reality: 0
Prediction: 0
Reality: 0